# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


Install the dependencies listed in the README before running this notebook.

In [1]:
import os
os.environ["USE_TF"] = "0"  # Use the PyTorch backend for Transformers.
import torch
torch.set_num_threads(2)
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [2]:
generator = pipeline("text-generation", model="distilbert/distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Device set to use cpu


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [4]:
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.999778687953949}]

# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

| Task | Family | Reason |
|---|---|---|
| Sentiment classification | Encoder-only | Both sides of a word help determine the sentiment of the whole sentence. |
| Machine translation | Encoder–decoder | The encoder reads the source and the decoder generates the target sentence. |
| Open-ended generation | Decoder-only | The model can extend a prompt one token at a time. |
| Named entity recognition | Encoder-only | Surrounding words help decide which tokens form a name. |
| Summarization | Encoder–decoder | The decoder can generate a shorter version conditioned on the source. |

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [5]:
sentences = ["The movie was excellent.", "The lecture was difficult to understand.",
             "I enjoyed learning about transformers."]
for sentence, result in zip(sentences, classifier(sentences)):
    print(f"{sentence}  {result['label']} ({result['score']:.3f})")

The movie was excellent.  POSITIVE (1.000)
The lecture was difficult to understand.  NEGATIVE (1.000)
I enjoyed learning about transformers.  POSITIVE (1.000)


## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [6]:
paragraph = ("A university library extended its opening hours during exams. "
             "Students can now use the study rooms until midnight. "
             "The library also hired two assistants to help during the evening shift.")
inputs = tokenizer("Summarize: " + paragraph, return_tensors="pt")
summary = model.generate(**inputs, max_new_tokens=40, num_beams=4)
print(tokenizer.decode(summary[0], skip_special_tokens=True))

Students at a university library have been able to use their study rooms until midnight.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


A decoder-only model generates a reply one token at a time. It uses the conversation and the words already generated to choose the next token. This lets it produce replies of different lengths and respond to follow-up questions in context.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [7]:
for result in generator("A helpful study habit is", do_sample=True, temperature=0.8,
                        max_new_tokens=20, num_return_sequences=3, pad_token_id=generator.tokenizer.eos_token_id):
    print(result["generated_text"])

A helpful study habit is an evolutionary process, it may be the basis of the evolutionary process itself. One of the most important
A helpful study habit is: It’s’t the first time you’ve seen a person tell you
A helpful study habit is part of the process, but one that seems to be more prevalent today.



There


Sampling draws from the next-token probability distribution. An early difference changes the context for every later draw, so the same prompt can lead to different continuations.